In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# ---------------------------------------------------------
# 0. Generate Synthetic Dataset matching requirements
# ---------------------------------------------------------
np.random.seed(42)
n_samples = 500

data = {
    "record_id": [f"REC_{i:04d}" for i in range(1, n_samples + 1)],
    "age": np.random.randint(18, 80, size=n_samples),
    "weight": np.random.normal(70, 15, size=n_samples).round(1),
    "gender": np.random.choice(["Male", "Female"], size=n_samples),
    "region": np.random.choice(["North", "South", "East", "West"], size=n_samples),
    "smoking_status": np.random.choice(["Smoker", "Non-Smoker", "Former Smoker"], size=n_samples, p=[0.3, 0.5, 0.2]),
    "exercise_frequency": np.random.choice(["Daily", "Weekly", "Rarely", "Never"], size=n_samples),
    "bmi": np.random.normal(25, 4, size=n_samples).round(1),
    "blood_pressure": np.random.normal(120, 15, size=n_samples).round(1),
    "cholesterol_level": np.random.normal(200, 30, size=n_samples).round(1),
    "glucose_level": np.random.normal(100, 25, size=n_samples).round(1),
}

df = pd.DataFrame(data)

# Derive categorical age_group
bins = [17, 25, 35, 45, 60, 100]
labels = ["18-25", "26-35", "36-45", "46-60", "60+"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)

# Derive disease variables based on risk factors
df["diabetes"] = (df["glucose_level"] > 115) | (df["bmi"] > 28)
df["hypertension"] = (df["blood_pressure"] > 130) | (df["age"] > 55)

# ---------------------------------------------------------
# 1. Hypotheses Formulation
# ---------------------------------------------------------
# Hypothesis 1:
# H0: Smoking status has no effect on Diabetes prevalence.
# H1: Smoking status affects Diabetes prevalence.

# Hypothesis 2:
# H0: Mean BMI is equal between Males and Females.
# H1: Mean BMI differs between Males and Females.

# ---------------------------------------------------------
# 2. Confidence Intervals (95%) for Numerical Variables
# ---------------------------------------------------------
def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), stats.sem(a)
    h = se * stats.t.ppf((1 + confidence) / 2., n-1)
    return m - h, m + h

age_ci = mean_confidence_interval(df["age"])
weight_ci = mean_confidence_interval(df["weight"])

print("--- 2. CONFIDENCE INTERVALS (95%) ---")
print(f"Age 95% CI: {age_ci[0]:.2f} to {age_ci[1]:.2f}")
print(f"Weight 95% CI: {weight_ci[0]:.2f} to {weight_ci[1]:.2f}\n")

# ---------------------------------------------------------
# 3, 4. Two-Sample t-test / Z-test (Mean BMI by Gender)
# ---------------------------------------------------------
male_bmi = df[df["gender"] == "Male"]["bmi"]
female_bmi = df[df["gender"] == "Female"]["bmi"]

t_stat, p_val_t = stats.ttest_ind(male_bmi, female_bmi)
critical_val_t = stats.t.ppf(1 - 0.05/2, df=len(df)-2)

print("--- 4. TWO-SAMPLE T-TEST (BMI vs Gender) ---")
print(f"t-statistic: {t_stat:.4f}")
print(f"Critical Value (alpha=0.05): {critical_val_t:.4f}")
print(f"p-value: {p_val_t:.4f}")
print(f"Decision: {'Reject H0' if p_val_t < 0.05 else 'Fail to Reject H0'}\n")

# ---------------------------------------------------------
# 5. Chi-Square Test (Smoking Status vs Diabetes)
# ---------------------------------------------------------
contingency_table = pd.crosstab(df["smoking_status"], df["diabetes"])
chi2, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table)
critical_val_chi2 = stats.chi2.ppf(0.95, dof)

print("--- 5. CHI-SQUARE TEST OF INDEPENDENCE ---")
print(f"Chi2 Statistic: {chi2:.4f}")
print(f"Degrees of Freedom: {dof}")
print(f"Critical Value (alpha=0.05): {critical_val_chi2:.4f}")
print(f"p-value: {p_val_chi2:.4f}")
print(f"Decision: {'Reject H0' if p_val_chi2 < 0.05 else 'Fail to Reject H0'}\n")

# ---------------------------------------------------------
# 6. One-Way ANOVA Test (Age Groups vs BMI)
# ---------------------------------------------------------
model = ols('bmi ~ C(age_group)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

f_stat = anova_table.loc['C(age_group)', 'F']
p_val_anova = anova_table.loc['C(age_group)', 'PR(>F)']

print("--- 6. ONE-WAY ANOVA TEST ---")
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value: {p_val_anova:.4f}")
print(f"Decision: {'Reject H0' if p_val_anova < 0.05 else 'Fail to Reject H0'}\n")

# ---------------------------------------------------------
# 7. Covariance and Correlation (Age vs BMI)
# ---------------------------------------------------------
cov_matrix = np.cov(df["age"], df["bmi"])[0][1]
corr, corr_p = stats.pearsonr(df["age"], df["bmi"])

print("--- 7. COVARIANCE AND CORRELATION ---")
print(f"Covariance (Age vs BMI): {cov_matrix:.4f}")
print(f"Pearson Correlation Coefficient: {corr:.4f} (p-value: {corr_p:.4f})")

--- 2. CONFIDENCE INTERVALS (95%) ---
Age 95% CI: 48.31 to 51.51
Weight 95% CI: 69.09 to 71.79

--- 4. TWO-SAMPLE T-TEST (BMI vs Gender) ---
t-statistic: -0.3563
Critical Value (alpha=0.05): 1.9647
p-value: 0.7217
Decision: Fail to Reject H0

--- 5. CHI-SQUARE TEST OF INDEPENDENCE ---
Chi2 Statistic: 0.2509
Degrees of Freedom: 2
Critical Value (alpha=0.05): 5.9915
p-value: 0.8821
Decision: Fail to Reject H0

--- 6. ONE-WAY ANOVA TEST ---
F-statistic: 0.6184
p-value: 0.6496
Decision: Fail to Reject H0

--- 7. COVARIANCE AND CORRELATION ---
Covariance (Age vs BMI): 1.5901
Pearson Correlation Coefficient: 0.0219 (p-value: 0.6258)
